# QURIOS — IA com redes tensoriais, na GPU grátis (Kaggle ou Google Colab)

Duas experiências, as duas com comparação justa (mesmo número de parâmetros, mesmos dados, mesmos passos):

**Parte 1 — encolher um modelo que já existe.** Qwen2.5-0.5B-Instruct (Apache-2.0, fala português): mede o original, comprime as camadas MLP de duas formas com **o mesmo tamanho** (posto baixo guiado pelos dados e rede tensorial TT), "cura" cada versão por destilação com textos e conversas, e compara qualidade, tamanho, velocidade e respostas lado a lado.

**Parte 2 — nascer em rede tensorial.** Um modelo de linguagem pequeno treinado do zero em quatro versões: denso, rede tensorial TT, posto baixo e denso estreito (as três últimas com o mesmo número de parâmetros). Na CPU, com largura 192, a TT venceu o posto baixo mas perdeu para o denso estreito; aqui a pergunta é se isso vira com camadas maiores.

Parte 1 leva cerca de 40 a 60 minutos numa T4; parte 2, perto de 1 hora.

**No Kaggle** (roda nos servidores dele, mesmo com o navegador fechado):
1. Em *Settings*, à direita: **Accelerator → GPU T4 x2** e **Internet → On** (precisa confirmar o telefone na conta Kaggle).
2. **Save Version → Save & Run All (Commit)**. O Kaggle dá 30 horas de GPU por semana.
3. No fim, na aba *Output* da versão: `resultado_llm.json` e `resultado_nativo.json`.

**No Google Colab** (se a fila de GPU do Kaggle estiver parada):
1. *Ambiente de execução → Alterar o tipo de ambiente de execução →* **GPU T4**.
2. *Ambiente de execução →* **Executar tudo**, e deixe a aba aberta até o fim.
3. Os dois JSON ficam em `/content` (ícone de pasta, à esquerda) e também saem impressos no fim de cada parte.

O QURIOS é instalado direto do GitHub público (github.com/matheusStark25/quirios-engine); não precisa subir nada. Nada aqui usa senha ou chave sua: os modelos e os textos são públicos.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# instala o QURIOS direto do GitHub público; se você subiu o .whl como dataset, usa ele
!WHL=$(find /kaggle/input -name "quirios_engine-*.whl" 2>/dev/null | head -n 1); if [ -n "$WHL" ]; then pip install -q "$WHL" "transformers>=4.45" datasets; else pip install -q "quirios-engine @ git+https://github.com/matheusStark25/quirios-engine.git" "transformers>=4.45" datasets; fi

In [ ]:
import json, os, torch
from quirios_engine import llm_tensorial as lt, llm_nativo as ln

# a mesma pasta de saída serve no Kaggle, no Colab ou no seu computador
if os.path.isdir("/kaggle/working"):
    PASTA = "/kaggle/working"
elif os.path.isdir("/content"):
    PASTA = "/content"
else:
    PASTA = os.getcwd()
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("dispositivo:", dispositivo, "| resultados em:", PASTA)


## Parte 1 — Qwen2.5-0.5B comprimido

A cura tem uma guarda: se a perda dispara, os pesos voltam ao último ponto bom e a taxa de aprendizado cai. O resultado registra quantas vezes isso aconteceu (`reducoes_lr`).


In [ ]:
MODELO = "Qwen/Qwen2.5-0.5B-Instruct"
FATOR = 1.5                      # cada camada MLP com 1,5x menos parâmetros
POSTO_TT = lt.posto_tt_equivalente(896, 4864, FATOR)   # TT com o mesmo tamanho do posto baixo
print("ligação TT:", POSTO_TT)
PERGUNTAS = ("Qual é a capital da França?",
             "Explique em uma frase o que é um computador quântico.",
             "Dê três dicas para dormir melhor.")

progresso = lambda m, s, l, dt: print(f"[{m}] cura passo {s}: perda {l:.3f} ({dt/60:.1f} min)", flush=True)
res = lt.demo(MODELO, metodos=("svd_dados", "tt"), posto=POSTO_TT, d=3, passos=2000, lote=4, seq=256,
              lr=3e-4, max_tokens=32768, fator=FATOR, dados_cura="misto", dispositivo=dispositivo,
              perguntas=PERGUNTAS, ao_progresso=progresso, saida=os.path.join(PASTA, "qurios_qwen"))
print(lt.texto_demo(res))
ARQ_LLM = os.path.join(PASTA, "resultado_llm.json")
with open(ARQ_LLM, "w", encoding="utf-8") as f:
    json.dump(res, f, ensure_ascii=False, indent=1)
print("salvo:", ARQ_LLM)
# o resultado completo também fica impresso, para não depender de baixar arquivo
print(json.dumps(res, ensure_ascii=False, default=str))


## Parte 2 — modelo nativo em rede tensorial, maior que o da CPU


In [ ]:
import gc; gc.collect(); torch.cuda.empty_cache() if dispositivo == "cuda" else None
mostra = lambda l: print(f"{l['modelo']}: {l['parametros']:,} parâm., {l['bits_por_caractere']:.3f} bits/caractere "
                         f"({l['s_treino']/60:.1f} min)", flush=True)
nat = ln.experimento(passos=5000, largura=384, camadas=6, cabecas=6, contexto=256, posto=24, lote=32,
                     dispositivo=dispositivo, ao_progresso=mostra)
print(ln.texto_experimento(nat))
ARQ_NATIVO = os.path.join(PASTA, "resultado_nativo.json")
with open(ARQ_NATIVO, "w", encoding="utf-8") as f:
    json.dump(nat, f, ensure_ascii=False, indent=1)
print("salvo:", ARQ_NATIVO)
print(json.dumps(nat, ensure_ascii=False, default=str))
